# Company Default Data

In [ ]:
# Importing the libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns # for making plots with seaborn
color = sns.color_palette()
import sklearn.metrics as metrics

import warnings
warnings.filterwarnings("ignore")

Let us now go ahead and read the dataset and check the first five rows of the dataset.

#### Importing the dataset

In [ ]:
Company = pd.read_csv('Company data.csv')

#Glimpse of Data
Company.head()

#### Fixing messy column names (containing spaces) for ease of use

In [ ]:
Company.columns = Company.columns.str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.replace('%', 'perc').str.replace('/', '_to_')

#### Checking top 5 rows again

In [ ]:
Company.head()

#### Now, let us check the number of rows (observations) and the number of columns (variables)

In [ ]:
print('The number of rows (observations) is',Company.shape[0],'\n''The number of columns (variables) is',Company.shape[1])

#### Checking datatype of all columns

In [ ]:
Company.info()

#### Now, let us check the basic measures of descriptive statistics for the continuous variables

In [ ]:
Company.describe()

In [ ]:
pd.options.display.float_format = '{:.2f}'.format   

Company.describe()

#### Creating a binary target variable using 'Networth_Next_Year' 

In [ ]:
Company['default'] = np.where((Company['Networth_Next_Year'] > 0), 0, 1)

#### Checking top 10 rows

In [ ]:
Company[['default','Networth_Next_Year']].head(10)

#### What does variable 'default' look like

In [ ]:
Company['default'].value_counts()

#### Checking proportion of default

In [ ]:
Company['default'].value_counts(normalize = True)

#### Lets check for missing values in the dataset

In [ ]:
Company.isnull().sum()

In [ ]:
Company.size

In [ ]:
Company.isnull().sum().sum()

There are missing values in the dataset

In [ ]:
Company_X = Company.drop('default', axis = 1)
Company_Y = Company['default']

In [ ]:
Company_X

#### Let's check the number of outliers per column

In [ ]:
Q1 = Company_X.quantile(0.25)
Q3 = Company_X.quantile(0.75)
IQR = Q3 - Q1
UL = Q3 + 1.5*IQR
LL = Q1 - 1.5*IQR

In [ ]:
((Company_X > UL) | (Company_X < LL)).sum()

In [ ]:
Company_X[((Company_X > UL) | (Company_X < LL))]= np.nan

In [ ]:
Company_X.isnull().sum()

In [ ]:
Company_X.isnull().sum().sum()

In [ ]:
Company_X = Company_X.drop(['Num', 'Networth_Next_Year', 'Equity_face_value'], axis = 1)

In [ ]:
Company_X.shape

In [ ]:
Company_sub1 = pd.concat([Company_X, Company_Y], axis =1 )

#### Let's visually inspect the missing values in our data

In [ ]:
plt.figure(figsize = (12,8))
sns.heatmap(Company_sub1.isnull(), cbar = False, cmap = 'coolwarm', yticklabels = False)
plt.show()

#### We should inspect total missing values by each row.

In [ ]:
Company_sub1.isnull().sum(axis = 1)

In [ ]:
Company_sub1_temp = Company_sub1[Company_sub1.isnull().sum(axis = 1) <= 5]

In [ ]:
Company_sub1_temp.shape

In [ ]:
Company_sub1_temp['default'].value_counts()

In [ ]:
19/243

If we consider availability of features for deciding the observations to be considered, we will end up losing more than 90% of the actual defaulters.

In [ ]:
Company_sub1['default'].value_counts()

In [ ]:
Company_sub1.isnull().sum().sort_values(ascending = False)/Company_sub1.index.size

#### Dropping columns with more than 30% missing values 

In [ ]:
Company_sub2 = Company_sub1.drop(['Deposits_accepted_by_commercial_banks', 'PE_on_BSE', 
                             'Investments', 'Other_income', 'Contingent_liabilities', 
                             'Deferred_tax_liability', 'Income_from_financial_services',
                                  'Shares_outstanding'],
                           axis = 1)

In [ ]:
Company_sub2.shape

#### Segregate the predictors and response 

In [ ]:
predictors = Company_sub2.drop('default', axis = 1)
response = Company_sub2['default']

#### Scale the predictors 

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_predictors = pd.DataFrame(scaler.fit_transform(predictors), columns = predictors.columns)

In [ ]:
Company_sub3 = pd.concat([scaled_predictors, response], axis = 1)

#### Imputing the remaining missing values

In [ ]:
from sklearn.impute import KNNImputer

In [ ]:
imputer = KNNImputer(n_neighbors=10)

In [ ]:
Company_imputed = pd.DataFrame(imputer.fit_transform(Company_sub3), columns = Company_sub3.columns)

In [ ]:
Company_imputed.isnull().sum()

#### Inspect possible correlations between independent variables 

In [ ]:
plt.figure(figsize = (12,8))
cor_matrix = Company_imputed.drop('default', axis = 1).corr()
sns.heatmap(cor_matrix, cmap = 'plasma', vmin = -1, vmax= 1)

In [ ]:
predictors = Company_imputed.drop('default', axis = 1)
response = Company_imputed['default']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

#### Splitting the data into train and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(predictors, response, 
                                                    test_size = 0.3, random_state = 2)

#### For modeling we will use Logistic Regression with recursive feature elimination

In [ ]:
LogR = LogisticRegression()

In [ ]:
selector = RFE(estimator = LogR, n_features_to_select=15, step=1)

In [ ]:
selector = selector.fit(X_train, y_train)

In [ ]:
selector.n_features_

In [ ]:
selector.ranking_

In [ ]:
df = pd.DataFrame({'Feature': scaled_predictors.columns, 'Rank': selector.ranking_})
df[df['Rank'] == 1]

#### Validating the model on train and test set 

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
pred_train = selector.predict(X_train)
pred_test = selector.predict(X_test)

In [ ]:
print(confusion_matrix(y_train, pred_train))

In [ ]:
print(classification_report(y_train, pred_train))

In [ ]:
print(confusion_matrix(y_test, pred_test))

In [ ]:
print(classification_report(y_test, pred_test))

We see poor recall score for both train and test

Since only 7% of the total data had defaults, we will now try to balance the data before fiting the model. 

In [ ]:
from imblearn.over_sampling import SMOTE 
sm = SMOTE(random_state=33)
X_res, y_res = sm.fit_resample(X_train, y_train)

In [ ]:
selector_smote = selector.fit(X_res, y_res)

In [ ]:
selector_smote.n_features_

In [ ]:
pred_train_smote = selector_smote.predict(X_res)
pred_test_smote = selector_smote.predict(X_test)

In [ ]:
print(classification_report(y_res, pred_train_smote))

In [ ]:
print(classification_report(y_test, pred_test_smote))

Finally, we are able to achieve a descent recall value without overfitting. Considering the opportunities such as outliers, missing values and correlated features this is a fairly good model. It can be improved if we get better quality data where the features explaining the default are not missing to this extent. Of course we can try other techniques which are not sensitive towards missing values and outliers.

## END